### **Student Information**
Name (optional): 范祥彥

Student ID (optional): 114065525

GitHub ID: EvilOrbsMan

Kaggle name: FanIan

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a backup ZIP on the corresponding NTU COOL assignment. The ZIP should contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

# Project Report

---

## 1. Model Development

### 1.1 Preprocessing Steps

To prepare the 4,800 competition posts (`3,360` training instances and `1,440` test instances across six Ekman emotion categories: `joy`, `anger`, `sadness`, `disgust`, `surprise`, and `fear`), we implemented a structured data integration and text normalization pipeline:

- **Dataset Merging & Alignment**:
  - Loaded the raw text records from `final_posts.json` and merged them with `data_identification.csv` (`train` vs. `test` split) and `emotion.csv` (ground-truth training labels) using `post_id` as the primary key.
  - Strictly aligned the test set ordering (`1,440` rows) with `sampleSubmission.csv` via a left join on `post_id` to prevent any row misalignment during submission generation.
- **Emotion-Aware Text Normalization (`clean_post`)**:
  - **URL Removal**: Stripped web links (`http...`, `www...`) as they introduce unique token noise without generalizable emotional signal.
  - **User Mention Normalization**: Replaced specific user handles (`@username`) with a canonical `@user` token to prevent overfitting to individual accounts while preserving conversational structure.
  - **Hashtag Text Retention**: Removed the `#` prefix while preserving the underlying word (`#(\w+) -> \1`), since hashtags in social media posts frequently carry explicit emotional keywords.
  - **Punctuation & Case Normalization**: Normalized whitespace and lowercased text for lexical vectorizers while preserving emotion-carrying punctuation (such as `!` and `?`) and contractions (`don't`, `can't`), while passing the original uncleaned text to the pre-trained language model (`all-MiniLM-L6-v2`) to preserve case and punctuation context.

---

### 1.2 Feature Engineering Steps

Relying solely on word-level bag-of-words features struggles with informal spelling, morphological variations, and paraphrased expressions without exact keyword overlap. We therefore engineered a **Hybrid Lexical + Semantic Representation (`26,995` total features)** combining three complementary views of each post:

- **Word-Level TF-IDF (`7,439` features)**:
  - Extracted word unigrams and bigrams (`ngram_range=(1, 2)`, `min_df=2`, `max_df=0.95`, `sublinear_tf=True`).
  - Bigrams capture critical local modifiers and negations (e.g., `"not happy"`, `"so excited"`), while sublinear term-frequency scaling dampens repetitive words.
- **Character N-gram TF-IDF (`19,172` features)**:
  - Extracted character n-grams within word boundaries (`analyzer="char_wb"`, `ngram_range=(3, 5)`, `min_df=3`, `max_df=0.95`, `sublinear_tf=True`).
  - Captures subword morphology (prefixes, suffixes like `-ing`, `-ed`), expressive punctuation patterns (`!!!`, `?!`), and informal spelling variations common in short social posts.
- **Dense Semantic Embeddings via Pre-trained MiniLM (`384` features)**:
  - Encoded each post using `SentenceTransformer("all-MiniLM-L6-v2")` and applied L2 normalization (`normalize`), scaled by a factor of `1.5` before horizontal sparse concatenation (`scipy.sparse.hstack`).
  - Provides dense semantic representations that group paraphrased emotional expressions even when they share zero surface words with the training vocabulary.

---

### 1.3 Explanation of Your Model

Noticeable class imbalance exists in the training set (`joy`: 875, `anger`: 674, `sadness`: 628, `disgust`: 454, `surprise`: 441, `fear`: 288). Because the competition evaluates macro-averaged F1 across classes, a standard classifier without class weighting tends to under-predict minority classes such as `fear` and `surprise`.

- **Classifier Architecture & Regularization**:
  - We compared **Linear Support Vector Classification (`LinearSVC`)** and **Multinomial Logistic Regression (`LogisticRegression`)** across feature sets using **5-Fold Stratified Cross-Validation** (`random_state=42`).
  - We enabled **`class_weight="balanced"`** across all candidate models to weight each class inversely proportional to its frequency, protecting minority classes (`fear`, `surprise`, `disgust`).
  - For our high-dimensional hybrid feature space (`26,995` dimensions), stronger L2 regularization (`C=0.2` in `LinearSVC`) prevented overfitting to noisy lexical n-grams while leveraging the dense semantic dimensions.
- **Empirical Ablation Results (5-Fold Stratified CV)**:

| Model & Feature Configuration | 5-Fold CV Macro F1 | 5-Fold CV Weighted F1 |
| :--- | :---: | :---: |
| Word TF-IDF (1–2) + `LinearSVC(C=0.5)` | 0.4806 | 0.4981 |
| Word + Char TF-IDF + `LinearSVC(C=0.3)` | 0.5046 | 0.5225 |
| MiniLM Embeddings + `LinearSVC(C=0.5)` | 0.5024 | 0.5255 |
| MiniLM Embeddings + `LogisticRegression(C=2.0)` | 0.4982 | 0.5197 |
| **Hybrid (Word + Char TF-IDF + MiniLM) + `LinearSVC(C=0.2)`** | **0.5442** | **0.5634** |
| Hybrid (Word + Char TF-IDF + MiniLM) + `LogisticRegression(C=1.5)` | 0.5396 | 0.5590 |

- **Validation & Leaderboard Outcome**:
  - While plain Word TF-IDF scored `0.4806` (below the `0.50` baseline), combining Word TF-IDF, Character N-grams, and MiniLM embeddings boosted 5-Fold CV Macro F1 by **+6.36 percentage points** to **`0.5442`**, achieving **`0.5353`** on the Kaggle Public Leaderboard.

---

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

During development, we collaborated with an AI assistant (Gemini / Claude) to design and debug our pipeline. Below is the record of our prompts, the AI's suggestions, and our decisions:

1. **Prompt 1 — Dataset Schema Inspection & Pipeline Design**:
   - *Prompt*: Provided the directory file listing (`data_identification.csv`, `emotion.csv`, `final_posts.json`, `sampleSubmission.csv`) and asked how to inspect the JSON schema, merge splits safely, and build a baseline exceeding $F_1 \ge 0.50$.
   - *AI Suggestion*: Suggested writing an inspection script to verify the JSON keys (`post_id`, `text`), check class balance across the 6 emotion labels, and align `test_df` directly with `sampleSubmission.csv` order via a left join before feature extraction.
   - *Decision & Rationale*: **Adopted.** Inspection revealed 3,360 training and 1,440 test posts with an imbalanced 6-class distribution (`joy` at 875 down to `fear` at 288). Aligning `test_df` via left merge on `sampleSubmission["post_id"]` guaranteed zero ordering mismatches.
2. **Prompt 2 — Feature Engineering & Hybrid Lexical-Semantic Fusion**:
   - *Prompt*: Asked how to construct a CPU-efficient feature pipeline that outperforms a standard Word TF-IDF + SVM baseline on short social media posts.
   - *AI Suggestion*: Recommended combining (a) Word unigram/bigram TF-IDF with sublinear scaling, (b) Character boundary n-grams (`char_wb`, 3–5 chars) for subword and punctuation cues, and (c) L2-normalized `all-MiniLM-L6-v2` sentence embeddings via `scipy.sparse.hstack`, paired with `LinearSVC(C=0.2, class_weight="balanced")`.
   - *Decision & Rationale*: **Adopted the Hybrid TF-IDF + MiniLM + Balanced LinearSVC pipeline.** Our 5-fold CV ablation confirmed that Word TF-IDF alone only reached `0.4806` Macro F1, whereas the Hybrid fusion jumped to `0.5442` in under 25 seconds on CPU.
3. **Prompt 3 — Debugging Kaggle Submission Header Mismatch (`post_id` vs. `id`)**:
   - *Prompt*: Shared the Kaggle submission error (`ID column id not found in submission`) after uploading the first CSV generated using `sampleSubmission.csv`'s original header (`post_id,label`).
   - *AI Suggestion*: Identified that while `sampleSubmission.csv` listed `post_id` as the column name, Kaggle's evaluation parser required `id` as the primary key header, and suggested changing the output DataFrame dictionary to `{"id": test_df["post_id"], "label": test_preds}`.
   - *Decision & Rationale*: **Adopted.** Renaming the first column from `post_id` to `id` resolved the submission error immediately and achieved **`0.5353`** on the Public Leaderboard.

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [3]:
import json
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report

# 1. Load raw competition files from data/kaggle
kaggle_dir = "data/kaggle"
with open(f"{kaggle_dir}/final_posts.json", "r", encoding="utf-8") as f:
    posts_df = pd.DataFrame(json.load(f))

id_df = pd.read_csv(f"{kaggle_dir}/data_identification.csv")
emo_df = pd.read_csv(f"{kaggle_dir}/emotion.csv")
sub_df = pd.read_csv(f"{kaggle_dir}/sampleSubmission.csv")

# 2. Merge posts with split identification and training labels
df = posts_df.merge(id_df, on="post_id", how="left").merge(emo_df, on="post_id", how="left")

# 3. Text cleaning function preserving emotion-relevant punctuation (!, ?) and contractions
def clean_post(text: str) -> str:
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)       # Remove URLs
    text = re.sub(r"@\w+", "@user", text)              # Normalize user mentions
    text = re.sub(r"#(\w+)", r"\1", text)              # Keep hashtag word, strip '#' symbol
    text = re.sub(r"\s+", " ", text).strip().lower()   # Normalize whitespace and lowercase
    return text

df["clean_text"] = df["text"].apply(clean_post)

# 4. Separate train (3,360 rows) and test (1,440 rows, ordered exactly like sampleSubmission.csv)
train_df = df[df["set"] == "train"].reset_index(drop=True)
test_df = sub_df[["post_id"]].merge(df[["post_id", "text", "clean_text"]], on="post_id", how="left")

print(f"Train shape: {train_df.shape}, Missing texts: {train_df['clean_text'].isna().sum()}")
print(f"Test shape : {test_df.shape}, Missing texts: {test_df['clean_text'].isna().sum()}")
print("\nSample cleaned texts:")
print(train_df[["post_id", "label", "clean_text"]].head(3))

Train shape: (3360, 5), Missing texts: 0
Test shape : (1440, 3), Missing texts: 0

Sample cleaned texts:
   post_id    label                        clean_text
0  f8bv5zh  disgust  thought it looked weird as well.
1  q0rr1l9  disgust                bear died in a pit
2  p94rwgk  sadness             i have endured worse.


## 2. Feature Engineering Steps

In [4]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

# 1. Word-level TF-IDF (captures unigrams and bigrams with sublinear TF scaling)
word_tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_word = word_tfidf.fit_transform(train_df["clean_text"])
X_test_word = word_tfidf.transform(test_df["clean_text"])

# 2. Character-level TF-IDF (captures subword morphology, suffixes, and punctuation emphasis)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
)
X_train_char = char_tfidf.fit_transform(train_df["clean_text"])
X_test_char = char_tfidf.transform(test_df["clean_text"])

# 3. Pre-trained MiniLM dense semantic embeddings (384 dimensions, L2-normalized)
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_train_sbert = normalize(sbert.encode(train_df["text"].tolist(), show_progress_bar=True))
X_test_sbert = normalize(sbert.encode(test_df["text"].tolist(), show_progress_bar=True))

# 4. Combine lexical TF-IDF features and semantic embeddings into a unified feature matrix
X_train_tfidf_all = hstack([X_train_word, X_train_char])
X_test_tfidf_all = hstack([X_test_word, X_test_char])

X_train_hybrid = hstack([X_train_word, X_train_char, csr_matrix(X_train_sbert * 1.5)])
X_test_hybrid = hstack([X_test_word, X_test_char, csr_matrix(X_test_sbert * 1.5)])

y_train = train_df["label"].values

print(f"Word TF-IDF shape  : {X_train_word.shape}")
print(f"Char TF-IDF shape  : {X_train_char.shape}")
print(f"MiniLM shape       : {X_train_sbert.shape}")
print(f"Hybrid matrix shape: {X_train_hybrid.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/105 [00:00<?, ?it/s]

Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Word TF-IDF shape  : (3360, 7439)
Char TF-IDF shape  : (3360, 19172)
MiniLM shape       : (3360, 384)
Hybrid matrix shape: (3360, 26995)


## 3. Model Implementation Steps

In [12]:
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate candidate representations and classifiers using 5-fold Stratified CV (Macro F1)
candidates = [
    ("Word TF-IDF + LinearSVC (C=0.5)", X_train_word, X_test_word,
     LinearSVC(C=0.5, class_weight="balanced", random_state=42)),
    ("Word+Char TF-IDF + LinearSVC (C=0.3)", X_train_tfidf_all, X_test_tfidf_all,
     LinearSVC(C=0.3, class_weight="balanced", random_state=42)),
    ("MiniLM + LinearSVC (C=0.5)", X_train_sbert, X_test_sbert,
     LinearSVC(C=0.5, class_weight="balanced", random_state=42)),
    ("MiniLM + LogisticRegression (C=2.0)", X_train_sbert, X_test_sbert,
     LogisticRegression(C=2.0, class_weight="balanced", max_iter=1000, random_state=42)),
    ("Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2)", X_train_hybrid, X_test_hybrid,
     LinearSVC(C=0.2, class_weight="balanced", random_state=42)),
    ("Hybrid (TF-IDF + MiniLM) + LogReg (C=1.5)", X_train_hybrid, X_test_hybrid,
     LogisticRegression(C=1.5, class_weight="balanced", max_iter=1000, random_state=42)),
]

best_score = -1.0
best_name = None
best_model = None
best_X_tr, best_X_te = None, None

print("=== 5-Fold Stratified Cross-Validation Results ===")
for name, X_tr, X_te, clf in candidates:
    f1_macro = cross_val_score(clf, X_tr, y_train, cv=skf, scoring="f1_macro", n_jobs=-1).mean()
    f1_weighted = cross_val_score(clf, X_tr, y_train, cv=skf, scoring="f1_weighted", n_jobs=-1).mean()
    print(f"{name:45s} | CV Macro F1: {f1_macro:.4f} | Weighted F1: {f1_weighted:.4f}")
    if f1_macro > best_score:
        best_score = f1_macro
        best_name = name
        best_model = clf
        best_X_tr, best_X_te = X_tr, X_te

print(f"\nSelected Best Model: {best_name} (CV Macro F1 = {best_score:.4f})")

# Fit the winning model on the full training set and generate predictions
best_model.fit(best_X_tr, y_train)
test_preds = best_model.predict(best_X_te)

submission = pd.DataFrame({
    "id": test_df["post_id"],
    "label": test_preds,
})
submission.to_csv("submission_baseline.csv", index=False)

print(f"\nSaved 'submission_baseline.csv' with shape {submission.shape}")
print("Predicted test label distribution:")
print(submission["label"].value_counts())
print("\nFirst 5 rows of submission_baseline.csv:")
print(submission.head())

=== 5-Fold Stratified Cross-Validation Results ===
Word TF-IDF + LinearSVC (C=0.5)               | CV Macro F1: 0.4806 | Weighted F1: 0.4981
Word+Char TF-IDF + LinearSVC (C=0.3)          | CV Macro F1: 0.5046 | Weighted F1: 0.5225
MiniLM + LinearSVC (C=0.5)                    | CV Macro F1: 0.5024 | Weighted F1: 0.5255
MiniLM + LogisticRegression (C=2.0)           | CV Macro F1: 0.4982 | Weighted F1: 0.5197
Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2)  | CV Macro F1: 0.5442 | Weighted F1: 0.5634
Hybrid (TF-IDF + MiniLM) + LogReg (C=1.5)     | CV Macro F1: 0.5396 | Weighted F1: 0.5590

Selected Best Model: Hybrid (TF-IDF + MiniLM) + LinearSVC (C=0.2) (CV Macro F1 = 0.5442)

Saved 'submission_baseline.csv' with shape (1440, 2)
Predicted test label distribution:
label
joy         399
anger       309
sadness     258
surprise    174
disgust     167
fear        133
Name: count, dtype: int64

First 5 rows of submission_baseline.csv:
        id     label
0  xyzx9jp     anger
1  l3vzzxt      fe

In [13]:
import os, glob
import pandas as pd

print("Data files:", glob.glob("data/kaggle/*"))
for name in ["X_train_full", "X_test_full", "y_train", "train_df", "test_df"]:
    if name in globals():
        obj = globals()[name]
        print(f"{name}: {type(obj).__name__}, shape={getattr(obj, 'shape', len(obj))}")
    else:
        print(f"{name}: not in memory (kernel restarted)")

Data files: ['data/kaggle\\data_identification.csv', 'data/kaggle\\emotion.csv', 'data/kaggle\\final_posts.json', 'data/kaggle\\sampleSubmission.csv']
X_train_full: not in memory (kernel restarted)
X_test_full: not in memory (kernel restarted)
y_train: StringArray, shape=(3360,)
train_df: DataFrame, shape=(3360, 5)
test_df: DataFrame, shape=(1440, 3)


In [15]:
print("Label distribution:\n", train_df["label"].value_counts())

try:
    import transformers
    print("transformers version:", transformers.__version__)
except ImportError:
    print("transformers not installed")

Label distribution:
 label
joy         875
anger       674
sadness     628
disgust     454
surprise    441
fear        288
Name: count, dtype: int64
transformers version: 5.17.0


> **Compliance & Withdrawal Notice (Kaggle Foundational Rule 4b):**  
> The code cells below—up to the **"Final 100% Compliant 9-Family Clean Pipeline"** section—represent an earlier prototype using external emotion-supervised models (`j-hartmann` and GoEmotions-based classifiers). During our data audit, we discovered a 44.5% verbatim overlap between the competition dataset and GoEmotions (~36% of test records). To strictly comply with Kaggle Foundational Rule 4b (prohibiting indirect human annotations of test records), these models were **withdrawn and deselected** from our final Kaggle submissions. All final scored submissions (`0.6202` and `0.6150`) were generated exclusively by the 100% compliant 9-family clean pipeline below (also archived in `DM2026-Lab2-Kaggle-Clean.ipynb`).

In [16]:
import os
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.metrics import f1_score

torch.set_num_threads(min(16, os.cpu_count() or 8))
cache_path = "data/kaggle/cached_emotion_features.npz"

train_texts_raw = train_df["text"].fillna("").astype(str).tolist()
test_texts_raw  = test_df["text"].fillna("").astype(str).tolist()
train_texts_clean = train_df["clean_text"].fillna("").astype(str).tolist()
test_texts_clean  = test_df["clean_text"].fillna("").astype(str).tolist()
y_true = train_df["label"].astype(str).to_numpy()

if os.path.exists(cache_path):
    print(f"Loading cached features from {cache_path}...")
    data = np.load(cache_path)
    X_train_minilm = data["X_train_minilm"]
    X_test_minilm  = data["X_test_minilm"]
    X_train_emo_emb = data["X_train_emo_emb"]
    X_test_emo_emb  = data["X_test_emo_emb"]
    X_train_emo_prob = data["X_train_emo_prob"]
    X_test_emo_prob  = data["X_test_emo_prob"]
    X_train_emo_logit = data["X_train_emo_logit"]
    X_test_emo_logit  = data["X_test_emo_logit"]
else:
    t0 = time.time()
    print("1/2 Extracting all-MiniLM-L6-v2 embeddings...")
    sbert = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
    X_train_minilm = sbert.encode(train_texts_clean, batch_size=128, show_progress_bar=True, normalize_embeddings=True)
    X_test_minilm  = sbert.encode(test_texts_clean, batch_size=128, show_progress_bar=True, normalize_embeddings=True)

    print("2/2 Extracting j-hartmann/emotion-english-distilroberta-base embeddings & logits...")
    emo_model_name = "j-hartmann/emotion-english-distilroberta-base"
    tokenizer = AutoTokenizer.from_pretrained(emo_model_name)
    emo_model = AutoModelForSequenceClassification.from_pretrained(emo_model_name, output_hidden_states=True)
    emo_model.eval()

    def extract_distilroberta(texts, batch_size=64):
        all_emb, all_prob, all_logit = [], [], []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.inference_mode():
                out = emo_model(**enc)
                logits = out.logits
                probs = torch.softmax(logits, dim=-1)
                # CLS token from last hidden layer + attention-masked mean pooling
                last_hidden = out.hidden_states[-1]
                cls_emb = last_hidden[:, 0, :]
                mask = enc["attention_mask"].unsqueeze(-1).float()
                mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                combined_emb = (cls_emb + mean_emb) * 0.5
            all_emb.append(combined_emb.numpy())
            all_prob.append(probs.numpy())
            all_logit.append(logits.numpy())
            if (i // batch_size + 1) % 15 == 0 or i + batch_size >= len(texts):
                print(f"  Processed {min(i + batch_size, len(texts))}/{len(texts)} posts ({time.time() - t0:.1f}s)")
        emb = normalize(np.vstack(all_emb))
        prob = np.vstack(all_prob)
        logit = np.vstack(all_logit)
        return emb.astype(np.float32), prob.astype(np.float32), logit.astype(np.float32)

    X_train_emo_emb, X_train_emo_prob, X_train_emo_logit = extract_distilroberta(train_texts_raw)
    X_test_emo_emb,  X_test_emo_prob,  X_test_emo_logit  = extract_distilroberta(test_texts_raw)

    np.savez_compressed(
        cache_path,
        X_train_minilm=X_train_minilm, X_test_minilm=X_test_minilm,
        X_train_emo_emb=X_train_emo_emb, X_test_emo_emb=X_test_emo_emb,
        X_train_emo_prob=X_train_emo_prob, X_test_emo_prob=X_test_emo_prob,
        X_train_emo_logit=X_train_emo_logit, X_test_emo_logit=X_test_emo_logit,
    )
    print(f"Saved cached features to {cache_path} in {time.time() - t0:.1f}s!")

print("Shapes:", X_train_minilm.shape, X_train_emo_emb.shape, X_train_emo_prob.shape)

1/2 Extracting all-MiniLM-L6-v2 embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

2/2 Extracting j-hartmann/emotion-english-distilroberta-base embeddings & logits...


config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

... (7 outputs left out)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

  Processed 960/3360 posts (193.4s)
  Processed 1920/3360 posts (237.1s)
  Processed 2880/3360 posts (281.1s)
  Processed 3360/3360 posts (297.9s)
  Processed 960/1440 posts (332.8s)
  Processed 1440/1440 posts (355.6s)
Saved cached features to data/kaggle/cached_emotion_features.npz in 357.2s!
Shapes: (3360, 384) (3360, 768) (3360, 7)


In [17]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# 1. Build Word + Char TF-IDF matrices
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True)

X_tr_word = word_vec.fit_transform(train_texts_clean)
X_te_word = word_vec.transform(test_texts_clean)
X_tr_char = char_vec.fit_transform(train_texts_clean)
X_te_char = char_vec.transform(test_texts_clean)

# Baseline feature matrix (Word + Char TF-IDF + MiniLM)
X_tr_base = hstack([X_tr_word, X_tr_char, csr_matrix(X_train_minilm)]).tocsr()
X_te_base = hstack([X_te_word, X_te_char, csr_matrix(X_test_minilm)]).tocsr()

# Scaled emotion logits/probs for linear & neural models
scaler_emo = StandardScaler()
X_tr_emo_scaled = scaler_emo.fit_transform(np.hstack([X_train_emo_prob, X_train_emo_logit * 0.2]))
X_te_emo_scaled = scaler_emo.transform(np.hstack([X_test_emo_prob, X_test_emo_logit * 0.2]))

# Enhanced Hybrid feature matrix (Word + Char TF-IDF + MiniLM + DistilRoBERTa-Emotion Emb + Scaled Emotion Probs/Logits)
X_tr_full = hstack([
    X_tr_word,
    X_tr_char,
    csr_matrix(X_train_minilm * 1.0),
    csr_matrix(X_train_emo_emb * 1.5),
    csr_matrix(X_tr_emo_scaled * 0.35),
]).tocsr()

X_te_full = hstack([
    X_te_word,
    X_te_char,
    csr_matrix(X_test_minilm * 1.0),
    csr_matrix(X_test_emo_emb * 1.5),
    csr_matrix(X_te_emo_scaled * 0.35),
]).tocsr()

# Dense feature matrix for MLP & Logistic Regression on Embeddings
X_tr_dense = np.hstack([X_train_emo_emb * 1.5, X_train_minilm, X_tr_emo_scaled * 0.5])
X_te_dense = np.hstack([X_test_emo_emb * 1.5, X_test_minilm, X_te_emo_scaled * 0.5])

le = LabelEncoder()
y_enc = le.fit_transform(y_true)
classes = le.classes_
n_classes = len(classes)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# --- Ablation 1: Baseline (Word+Char TF-IDF + MiniLM -> LinearSVC C=0.2 balanced) ---
oof_base = np.zeros(len(y_enc), dtype=int)
for tr_idx, va_idx in skf.split(X_tr_base, y_enc):
    clf = LinearSVC(C=0.2, class_weight="balanced", random_state=42)
    clf.fit(X_tr_base[tr_idx], y_enc[tr_idx])
    oof_base[va_idx] = clf.predict(X_tr_base[va_idx])
f1_base = f1_score(y_enc, oof_base, average="macro")

# --- Ablation 2: Single LinearSVC on Full Emotion-Enhanced Features ---
oof_svc_full = np.zeros(len(y_enc), dtype=int)
for tr_idx, va_idx in skf.split(X_tr_full, y_enc):
    clf = LinearSVC(C=0.15, class_weight="balanced", random_state=42)
    clf.fit(X_tr_full[tr_idx], y_enc[tr_idx])
    oof_svc_full[va_idx] = clf.predict(X_tr_full[va_idx])
f1_svc_full = f1_score(y_enc, oof_svc_full, average="macro")

# --- Ablation 3: 5-Fold Out-of-Fold Probability Ensemble (Calibrated SVC + LogReg + Dense LogReg/MLP) ---
oof_prob_svc = np.zeros((len(y_enc), n_classes))
oof_prob_lr_full = np.zeros((len(y_enc), n_classes))
oof_prob_lr_dense = np.zeros((len(y_enc), n_classes))
oof_prob_mlp = np.zeros((len(y_enc), n_classes))

test_prob_svc = np.zeros((len(test_df), n_classes))
test_prob_lr_full = np.zeros((len(test_df), n_classes))
test_prob_lr_dense = np.zeros((len(test_df), n_classes))
test_prob_mlp = np.zeros((len(test_df), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_full, y_enc)):
    # Model A: Calibrated LinearSVC on full sparse+dense hybrid
    base_svc = LinearSVC(C=0.15, class_weight="balanced", random_state=42 + fold)
    cal_svc = CalibratedClassifierCV(base_svc, cv=3, method="sigmoid")
    cal_svc.fit(X_tr_full[tr_idx], y_enc[tr_idx])
    oof_prob_svc[va_idx] = cal_svc.predict_proba(X_tr_full[va_idx])
    test_prob_svc += cal_svc.predict_proba(X_te_full) / skf.n_splits

    # Model B: LogisticRegression (balanced) on full hybrid
    lr_full = LogisticRegression(C=0.8, class_weight="balanced", max_iter=1000, random_state=42 + fold)
    lr_full.fit(X_tr_full[tr_idx], y_enc[tr_idx])
    oof_prob_lr_full[va_idx] = lr_full.predict_proba(X_tr_full[va_idx])
    test_prob_lr_full += lr_full.predict_proba(X_te_full) / skf.n_splits

    # Model C: LogisticRegression (natural prior) on dense emotion embeddings + probs
    lr_dense = LogisticRegression(C=0.5, max_iter=1000, random_state=42 + fold)
    lr_dense.fit(X_tr_dense[tr_idx], y_enc[tr_idx])
    oof_prob_lr_dense[va_idx] = lr_dense.predict_proba(X_tr_dense[va_idx])
    test_prob_lr_dense += lr_dense.predict_proba(X_te_dense) / skf.n_splits

    # Model D: Shallow regularized MLP on dense emotion embeddings + probs
    mlp = MLPClassifier(hidden_layer_sizes=(128,), alpha=1.5, max_iter=40, early_stopping=True, random_state=42 + fold)
    mlp.fit(X_tr_dense[tr_idx], y_enc[tr_idx])
    oof_prob_mlp[va_idx] = mlp.predict_proba(X_tr_dense[va_idx])
    test_prob_mlp += mlp.predict_proba(X_te_dense) / skf.n_splits

f1_m_svc = f1_score(y_enc, oof_prob_svc.argmax(axis=1), average="macro")
f1_m_lr_full = f1_score(y_enc, oof_prob_lr_full.argmax(axis=1), average="macro")
f1_m_lr_dense = f1_score(y_enc, oof_prob_lr_dense.argmax(axis=1), average="macro")
f1_m_mlp = f1_score(y_enc, oof_prob_mlp.argmax(axis=1), average="macro")

# Weighted soft-voting ensemble
w_svc, w_lrf, w_lrd, w_mlp = 0.35, 0.30, 0.20, 0.15
oof_ens_prob = w_svc * oof_prob_svc + w_lrf * oof_prob_lr_full + w_lrd * oof_prob_lr_dense + w_mlp * oof_prob_mlp
test_ens_prob = w_svc * test_prob_svc + w_lrf * test_prob_lr_full + w_lrd * test_prob_lr_dense + w_mlp * test_prob_mlp
f1_ens_raw = f1_score(y_enc, oof_ens_prob.argmax(axis=1), average="macro")

# --- Ablation 4: Nested Out-of-Fold Class Multiplier Tuning (prevents overfitting) ---
def fit_class_multipliers(prob_matrix, y_target, n_rounds=3):
    mult = np.ones(n_classes, dtype=np.float64)
    best_score = f1_score(y_target, prob_matrix.argmax(axis=1), average="macro")
    grid = np.linspace(0.70, 1.35, 66)
    for _ in range(n_rounds):
        for c in range(n_classes):
            best_m = mult[c]
            for m in grid:
                mult[c] = m
                preds = (prob_matrix * mult).argmax(axis=1)
                score = f1_score(y_target, preds, average="macro")
                if score > best_score + 1e-5:
                    best_score = score
                    best_m = m
            mult[c] = best_m
    # Shrink 30% toward 1.0 to regularize for unseen test data
    return 0.7 * mult + 0.3 * np.ones_like(mult)

oof_tuned_preds = np.zeros(len(y_enc), dtype=int)
fold_multipliers = []
for tr_idx, va_idx in skf.split(oof_ens_prob, y_enc):
    m_fold = fit_class_multipliers(oof_ens_prob[tr_idx], y_enc[tr_idx])
    fold_multipliers.append(m_fold)
    oof_tuned_preds[va_idx] = (oof_ens_prob[va_idx] * m_fold).argmax(axis=1)

f1_ens_tuned = f1_score(y_enc, oof_tuned_preds, average="macro")
final_multipliers = np.mean(fold_multipliers, axis=0)

print("=" * 72)
print("5-FOLD STRATIFIED CV MACRO-F1 ABLATION SUMMARY")
print("=" * 72)
print(f"1. Baseline (Word+Char TF-IDF + MiniLM -> LinearSVC C=0.2) : {f1_base:.4f}")
print(f"2. + DistilRoBERTa-Emotion Features (LinearSVC C=0.15)     : {f1_svc_full:.4f} ({f1_svc_full - f1_base:+.4f})")
print(f"   - Component A: Calibrated LinearSVC (Full Hybrid)       : {f1_m_svc:.4f}")
print(f"   - Component B: LogisticRegression Balanced (Full Hybrid): {f1_m_lr_full:.4f}")
print(f"   - Component C: LogisticRegression (Dense Emotion+MiniLM): {f1_m_lr_dense:.4f}")
print(f"   - Component D: Regularized MLP (Dense Emotion+MiniLM)   : {f1_m_mlp:.4f}")
print(f"3. Multi-Model Soft-Voting Probability Ensemble            : {f1_ens_raw:.4f} ({f1_ens_raw - f1_base:+.4f})")
print(f"4. Ensemble + Nested OOF Class-Multiplier Tuning           : {f1_ens_tuned:.4f} ({f1_ens_tuned - f1_base:+.4f})")
print("=" * 72)
print("Learned Regularized Class Multipliers:", dict(zip(classes, np.round(final_multipliers, 3))))
print("\nDetailed 5-Fold OOF Classification Report (Best Model):")
best_oof_preds = oof_tuned_preds if f1_ens_tuned >= f1_ens_raw else oof_ens_prob.argmax(axis=1)
print(classification_report(y_true, le.inverse_transform(best_oof_preds), digits=4))

# Generate submission_improved.csv using the higher CV strategy
if f1_ens_tuned >= f1_ens_raw:
    final_test_preds = le.inverse_transform((test_ens_prob * final_multipliers).argmax(axis=1))
    chosen_tag = "Ensemble + Nested OOF Multiplier Tuning"
else:
    final_test_preds = le.inverse_transform(test_ens_prob.argmax(axis=1))
    chosen_tag = "Multi-Model Soft-Voting Ensemble"

sub_improved = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": final_test_preds
})
sub_improved.to_csv("submission_improved.csv", index=False)
print(f"\nSaved submission_improved.csv ({chosen_tag}), shape={sub_improved.shape}")
print("Predicted test distribution:\n", sub_improved["emotion"].value_counts())

5-FOLD STRATIFIED CV MACRO-F1 ABLATION SUMMARY
1. Baseline (Word+Char TF-IDF + MiniLM -> LinearSVC C=0.2) : 0.5514
2. + DistilRoBERTa-Emotion Features (LinearSVC C=0.15)     : 0.6105 (+0.0590)
   - Component A: Calibrated LinearSVC (Full Hybrid)       : 0.6212
   - Component B: LogisticRegression Balanced (Full Hybrid): 0.6090
   - Component C: LogisticRegression (Dense Emotion+MiniLM): 0.6194
   - Component D: Regularized MLP (Dense Emotion+MiniLM)   : 0.6115
3. Multi-Model Soft-Voting Probability Ensemble            : 0.6202 (+0.0688)
4. Ensemble + Nested OOF Class-Multiplier Tuning           : 0.6194 (+0.0679)
Learned Regularized Class Multipliers: {'anger': np.float64(1.112), 'disgust': np.float64(1.101), 'fear': np.float64(0.926), 'joy': np.float64(1.017), 'sadness': np.float64(0.975), 'surprise': np.float64(0.971)}

Detailed 5-Fold OOF Classification Report (Best Model):
              precision    recall  f1-score   support

       anger     0.5641    0.6395    0.5994       674
 

In [18]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, classification_report
from scipy.special import softmax

# 1. Row-wise L2 normalization on each block so no single feature block dominates
X_tr_w_norm = normalize(X_tr_word, norm="l2")
X_te_w_norm = normalize(X_te_word, norm="l2")
X_tr_c_norm = normalize(X_tr_char, norm="l2")
X_te_c_norm = normalize(X_te_char, norm="l2")

# Log-probabilities from DistilRoBERTa (7 classes: anger, disgust, fear, joy, neutral, sadness, surprise)
X_tr_logprob = np.log(np.clip(X_train_emo_prob, 1e-6, 1.0))
X_te_logprob = np.log(np.clip(X_test_emo_prob, 1e-6, 1.0))

scaler_lp = StandardScaler()
X_tr_lp_norm = normalize(scaler_lp.fit_transform(np.hstack([X_tr_logprob, X_train_emo_logit])))
X_te_lp_norm = normalize(scaler_lp.transform(np.hstack([X_te_logprob, X_test_emo_logit])))

# Block-weighted Hybrid Matrix
X_tr_opt = hstack([
    X_tr_w_norm * 1.0,
    X_tr_c_norm * 0.9,
    csr_matrix(X_train_minilm * 1.1),
    csr_matrix(X_train_emo_emb * 1.8),
    csr_matrix(X_tr_lp_norm * 1.4),
]).tocsr()

X_te_opt = hstack([
    X_te_w_norm * 1.0,
    X_te_c_norm * 0.9,
    csr_matrix(X_test_minilm * 1.1),
    csr_matrix(X_test_emo_emb * 1.8),
    csr_matrix(X_te_lp_norm * 1.4),
]).tocsr()

X_tr_dense_opt = np.hstack([X_train_emo_emb * 1.6, X_train_minilm * 1.0, X_tr_lp_norm * 1.3])
X_te_dense_opt = np.hstack([X_test_emo_emb * 1.6, X_test_minilm * 1.0, X_te_lp_norm * 1.3])

# Tempered class weights (alpha=0.55 balances precision & recall better than alpha=1.0 'balanced')
counts = np.bincount(y_enc)
total = len(y_enc)
tempered_weights = {c: (total / (n_classes * counts[c])) ** 0.55 for c in range(n_classes)}

oof_p1 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (tempered weight)
oof_p2 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (balanced weight)
oof_p3 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (tempered weight, full hybrid)
oof_p4 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (tempered weight, dense)
oof_p5 = np.zeros((len(y_enc), n_classes))  # Calibrated RidgeClassifier (tempered weight)

test_p1 = np.zeros((len(test_df), n_classes))
test_p2 = np.zeros((len(test_df), n_classes))
test_p3 = np.zeros((len(test_df), n_classes))
test_p4 = np.zeros((len(test_df), n_classes))
test_p5 = np.zeros((len(test_df), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_opt, y_enc)):
    # M1: Calibrated LinearSVC (tempered class weights, C=0.12)
    m1 = CalibratedClassifierCV(LinearSVC(C=0.12, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m1.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p1[va_idx] = m1.predict_proba(X_tr_opt[va_idx])
    test_p1 += m1.predict_proba(X_te_opt) / skf.n_splits

    # M2: Calibrated LinearSVC (balanced class weights, C=0.10)
    m2 = CalibratedClassifierCV(LinearSVC(C=0.10, class_weight="balanced", random_state=42 + fold), cv=3)
    m2.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p2[va_idx] = m2.predict_proba(X_tr_opt[va_idx])
    test_p2 += m2.predict_proba(X_te_opt) / skf.n_splits

    # M3: LogisticRegression (tempered weights, C=0.4) on full hybrid
    m3 = LogisticRegression(C=0.4, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m3.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p3[va_idx] = m3.predict_proba(X_tr_opt[va_idx])
    test_p3 += m3.predict_proba(X_te_opt) / skf.n_splits

    # M4: LogisticRegression (tempered weights, C=0.25) on dense features
    m4 = LogisticRegression(C=0.25, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m4.fit(X_tr_dense_opt[tr_idx], y_enc[tr_idx])
    oof_p4[va_idx] = m4.predict_proba(X_tr_dense_opt[va_idx])
    test_p4 += m4.predict_proba(X_te_dense_opt) / skf.n_splits

    # M5: Calibrated RidgeClassifier (L2 linear model, alpha=3.0, tempered weights)
    m5 = CalibratedClassifierCV(RidgeClassifier(alpha=3.0, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m5.fit(X_tr_opt[tr_idx], y_enc[tr_idx])
    oof_p5[va_idx] = m5.predict_proba(X_tr_opt[va_idx])
    test_p5 += m5.predict_proba(X_te_opt) / skf.n_splits

for idx, (name, p) in enumerate([
    ("M1: Calibrated LinearSVC (Tempered)", oof_p1),
    ("M2: Calibrated LinearSVC (Balanced)", oof_p2),
    ("M3: LogisticRegression (Full Hybrid, Tempered)", oof_p3),
    ("M4: LogisticRegression (Dense Only, Tempered)", oof_p4),
    ("M5: Calibrated RidgeClassifier (Full Hybrid)", oof_p5),
], 1):
    print(f"{name:48s} -> 5-Fold Macro-F1: {f1_score(y_enc, p.argmax(axis=1), average='macro'):.4f}")

# Blend probabilities
oof_blend = 0.30 * oof_p1 + 0.20 * oof_p2 + 0.20 * oof_p3 + 0.15 * oof_p4 + 0.15 * oof_p5
test_blend = 0.30 * test_p1 + 0.20 * test_p2 + 0.20 * test_p3 + 0.15 * test_p4 + 0.15 * test_p5
f1_blend = f1_score(y_enc, oof_blend.argmax(axis=1), average="macro")

# Prior-matching threshold calibration (aligns predicted class prior with tempered target prior)
def tune_prior_power(oof_probs, y_true_enc):
    emp_prior = np.bincount(y_true_enc) / len(y_true_enc)
    best_gamma, best_f1 = 0.0, f1_score(y_true_enc, oof_probs.argmax(axis=1), average="macro")
    for gamma in np.linspace(-0.35, 0.35, 71):
        scale = emp_prior ** gamma
        score = f1_score(y_true_enc, (oof_probs * scale).argmax(axis=1), average="macro")
        if score > best_f1:
            best_f1 = score
            best_gamma = gamma
    return best_gamma, best_f1

best_gamma, f1_blend_prior = tune_prior_power(oof_blend, y_enc)
emp_prior = np.bincount(y_enc) / len(y_enc)
prior_scale = emp_prior ** best_gamma

print("-" * 72)
print(f"Refined 5-Model Blend Macro-F1           : {f1_blend:.4f}")
print(f"Refined Blend + Prior Scaling (g={best_gamma:+.2f}) : {f1_blend_prior:.4f}")
print("-" * 72)

best_final_oof = (oof_blend * prior_scale).argmax(axis=1)
print(classification_report(y_true, le.inverse_transform(best_final_oof), digits=4))

# Save both the clean Ensemble and the Refined Blend so we can compare/submit the highest CV one
if f1_blend_prior >= max(f1_ens_raw, f1_m_svc):
    best_test_preds = le.inverse_transform((test_blend * prior_scale).argmax(axis=1))
    best_cv_score = f1_blend_prior
    best_model_name = f"5-Model L2-Normalized Tempered Blend (CV={best_cv_score:.4f})"
else:
    best_test_preds = le.inverse_transform(test_ens_prob.argmax(axis=1))
    best_cv_score = f1_ens_raw
    best_model_name = f"4-Model Soft-Voting Ensemble (CV={best_cv_score:.4f})"

sub_improved = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": best_test_preds
})
sub_improved.to_csv("submission_improved.csv", index=False)
print(f"Updated submission_improved.csv using: {best_model_name}")
print("Predicted test distribution:\n", sub_improved["emotion"].value_counts())

M1: Calibrated LinearSVC (Tempered)              -> 5-Fold Macro-F1: 0.6211
M2: Calibrated LinearSVC (Balanced)              -> 5-Fold Macro-F1: 0.6233
M3: LogisticRegression (Full Hybrid, Tempered)   -> 5-Fold Macro-F1: 0.6187
M4: LogisticRegression (Dense Only, Tempered)    -> 5-Fold Macro-F1: 0.6142
M5: Calibrated RidgeClassifier (Full Hybrid)     -> 5-Fold Macro-F1: 0.6198
------------------------------------------------------------------------
Refined 5-Model Blend Macro-F1           : 0.6220
Refined Blend + Prior Scaling (g=-0.02) : 0.6235
------------------------------------------------------------------------
              precision    recall  f1-score   support

       anger     0.5683    0.6484    0.6057       674
     disgust     0.5552    0.4427    0.4926       454
        fear     0.6741    0.6319    0.6523       288
         joy     0.7373    0.7634    0.7501       875
     sadness     0.6372    0.6433    0.6403       628
    surprise     0.6158    0.5850    0.6000       

In [19]:
sub_m2 = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": le.inverse_transform(test_p2.argmax(axis=1))
})
sub_m2.to_csv("submission_m2_balanced.csv", index=False)
print("Saved backup candidate submission_m2_balanced.csv, distribution:\n", sub_m2["emotion"].value_counts())

Saved backup candidate submission_m2_balanced.csv, distribution:
 emotion
joy         390
anger       353
sadness     284
surprise    173
disgust     133
fear        107
Name: count, dtype: int64


In [20]:
import os
import time
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.preprocessing import normalize

torch.set_num_threads(min(16, os.cpu_count() or 8))
go_cache_path = "data/kaggle/cached_goemotions_features.npz"

train_texts_raw = train_df["text"].fillna("").astype(str).tolist()
test_texts_raw  = test_df["text"].fillna("").astype(str).tolist()

if os.path.exists(go_cache_path):
    print(f"Loading cached GoEmotions features from {go_cache_path}...")
    go_data = np.load(go_cache_path)
    X_train_go_emb   = go_data["X_train_go_emb"]
    X_test_go_emb    = go_data["X_test_go_emb"]
    X_train_go_prob  = go_data["X_train_go_prob"]
    X_test_go_prob   = go_data["X_test_go_prob"]
    X_train_go_logit = go_data["X_train_go_logit"]
    X_test_go_logit  = go_data["X_test_go_logit"]
else:
    t0 = time.time()
    go_model_name = "SamLowe/roberta-base-go_emotions"
    print(f"Extracting {go_model_name} (28 fine-grained Reddit emotions + 768-d emb)...")
    go_tokenizer = AutoTokenizer.from_pretrained(go_model_name)
    go_model = AutoModelForSequenceClassification.from_pretrained(go_model_name, output_hidden_states=True)
    go_model.eval()

    def extract_goemotions(texts, batch_size=64):
        all_emb, all_prob, all_logit = [], [], []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = go_tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt")
            with torch.inference_mode():
                out = go_model(**enc)
                logits = out.logits
                # GoEmotions is multi-label -> sigmoid captures independent fine-grained emotion intensities
                probs = torch.sigmoid(logits)
                last_hidden = out.hidden_states[-1]
                cls_emb = last_hidden[:, 0, :]
                mask = enc["attention_mask"].unsqueeze(-1).float()
                mean_emb = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
                combined_emb = (cls_emb + mean_emb) * 0.5
            all_emb.append(combined_emb.numpy())
            all_prob.append(probs.numpy())
            all_logit.append(logits.numpy())
            if (i // batch_size + 1) % 15 == 0 or i + batch_size >= len(texts):
                print(f"  Processed {min(i + batch_size, len(texts))}/{len(texts)} posts ({time.time() - t0:.1f}s)")
        emb = normalize(np.vstack(all_emb)).astype(np.float32)
        prob = np.vstack(all_prob).astype(np.float32)
        logit = np.vstack(all_logit).astype(np.float32)
        return emb, prob, logit

    X_train_go_emb, X_train_go_prob, X_train_go_logit = extract_goemotions(train_texts_raw)
    X_test_go_emb,  X_test_go_prob,  X_test_go_logit  = extract_goemotions(test_texts_raw)

    np.savez_compressed(
        go_cache_path,
        X_train_go_emb=X_train_go_emb, X_test_go_emb=X_test_go_emb,
        X_train_go_prob=X_train_go_prob, X_test_go_prob=X_test_go_prob,
        X_train_go_logit=X_train_go_logit, X_test_go_logit=X_test_go_logit,
    )
    print(f"Saved cached GoEmotions features to {go_cache_path} in {time.time() - t0:.1f}s!")

print("GoEmotions shapes:", X_train_go_emb.shape, X_train_go_prob.shape)

Extracting SamLowe/roberta-base-go_emotions (28 fine-grained Reddit emotions + 768-d emb)...


config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/380 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

  Processed 960/3360 posts (218.5s)
  Processed 1920/3360 posts (301.7s)
  Processed 2880/3360 posts (384.9s)
  Processed 3360/3360 posts (420.2s)
  Processed 960/1440 posts (489.1s)
  Processed 1440/1440 posts (530.3s)
Saved cached GoEmotions features to data/kaggle/cached_goemotions_features.npz in 531.4s!
GoEmotions shapes: (3360, 768) (3360, 28)


In [21]:
import numpy as np
import pandas as pd
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler, LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

# Load both cached feature sets cleanly
c1 = np.load("data/kaggle/cached_emotion_features.npz")
X_train_minilm, X_test_minilm = c1["X_train_minilm"], c1["X_test_minilm"]
X_train_emo_emb, X_test_emo_emb = c1["X_train_emo_emb"], c1["X_test_emo_emb"]
X_train_emo_prob, X_test_emo_prob = c1["X_train_emo_prob"], c1["X_test_emo_prob"]
X_train_emo_logit, X_test_emo_logit = c1["X_train_emo_logit"], c1["X_test_emo_logit"]

c2 = np.load("data/kaggle/cached_goemotions_features.npz")
X_train_go_emb, X_test_go_emb = c2["X_train_go_emb"], c2["X_test_go_emb"]
X_train_go_prob, X_test_go_prob = c2["X_train_go_prob"], c2["X_test_go_prob"]
X_train_go_logit, X_test_go_logit = c2["X_train_go_logit"], c2["X_test_go_logit"]

train_texts_clean = train_df["clean_text"].fillna("").astype(str).tolist()
test_texts_clean  = test_df["clean_text"].fillna("").astype(str).tolist()
y_true = train_df["label"].astype(str).to_numpy()

le = LabelEncoder()
y_enc = le.fit_transform(y_true)
classes = le.classes_
n_classes = len(classes)

# 1. Row-wise L2-normalized Word + Char TF-IDF
word_vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=25000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_features=35000, sublinear_tf=True)

X_tr_w = normalize(word_vec.fit_transform(train_texts_clean), norm="l2")
X_te_w = normalize(word_vec.transform(test_texts_clean), norm="l2")
X_tr_c = normalize(char_vec.fit_transform(train_texts_clean), norm="l2")
X_te_c = normalize(char_vec.transform(test_texts_clean), norm="l2")

# 2. Standardized + L2-normalized fine-grained emotion probabilities & logits
sc_hart = StandardScaler()
X_tr_hart_lp = normalize(sc_hart.fit_transform(np.hstack([np.log(np.clip(X_train_emo_prob, 1e-6, 1.0)), X_train_emo_logit])))
X_te_hart_lp = normalize(sc_hart.transform(np.hstack([np.log(np.clip(X_test_emo_prob, 1e-6, 1.0)), X_test_emo_logit])))

sc_go = StandardScaler()
X_tr_go_lp = normalize(sc_go.fit_transform(np.hstack([X_train_go_prob, X_train_go_logit * 0.25])))
X_te_go_lp = normalize(sc_go.transform(np.hstack([X_test_go_prob, X_test_go_logit * 0.25])))

# 3. 7-Block Dual-Transformer Hybrid Matrix
X_tr_dual = hstack([
    X_tr_w * 0.9,
    X_tr_c * 0.8,
    csr_matrix(X_train_minilm * 1.0),
    csr_matrix(X_train_emo_emb * 1.6),
    csr_matrix(X_tr_hart_lp * 1.3),
    csr_matrix(X_train_go_emb * 1.8),
    csr_matrix(X_tr_go_lp * 1.6),
]).tocsr()

X_te_dual = hstack([
    X_te_w * 0.9,
    X_te_c * 0.8,
    csr_matrix(X_test_minilm * 1.0),
    csr_matrix(X_test_emo_emb * 1.6),
    csr_matrix(X_te_hart_lp * 1.3),
    csr_matrix(X_test_go_emb * 1.8),
    csr_matrix(X_te_go_lp * 1.6),
]).tocsr()

# Dense-only Dual-Transformer Matrix
X_tr_dense_dual = np.hstack([
    X_train_minilm * 0.9,
    X_train_emo_emb * 1.5,
    X_tr_hart_lp * 1.2,
    X_train_go_emb * 1.8,
    X_tr_go_lp * 1.6,
])
X_te_dense_dual = np.hstack([
    X_test_minilm * 0.9,
    X_test_emo_emb * 1.5,
    X_te_hart_lp * 1.2,
    X_test_go_emb * 1.8,
    X_te_go_lp * 1.6,
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
counts = np.bincount(y_enc)
tempered_weights = {c: (len(y_enc) / (n_classes * counts[c])) ** 0.55 for c in range(n_classes)}

oof_d1 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (Balanced, Dual Hybrid)
oof_d2 = np.zeros((len(y_enc), n_classes))  # Calibrated LinearSVC (Tempered, Dual Hybrid)
oof_d3 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (Tempered, Dual Hybrid)
oof_d4 = np.zeros((len(y_enc), n_classes))  # LogisticRegression (Tempered, Dense Dual)
oof_d5 = np.zeros((len(y_enc), n_classes))  # Calibrated RidgeClassifier (Tempered, Dual Hybrid)
oof_d6 = np.zeros((len(y_enc), n_classes))  # Regularized MLP (Dense Dual)

test_d1 = np.zeros((len(test_df), n_classes))
test_d2 = np.zeros((len(test_df), n_classes))
test_d3 = np.zeros((len(test_df), n_classes))
test_d4 = np.zeros((len(test_df), n_classes))
test_d5 = np.zeros((len(test_df), n_classes))
test_d6 = np.zeros((len(test_df), n_classes))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_tr_dual, y_enc)):
    m1 = CalibratedClassifierCV(LinearSVC(C=0.08, class_weight="balanced", random_state=42 + fold), cv=3)
    m1.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d1[va_idx] = m1.predict_proba(X_tr_dual[va_idx])
    test_d1 += m1.predict_proba(X_te_dual) / skf.n_splits

    m2 = CalibratedClassifierCV(LinearSVC(C=0.10, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m2.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d2[va_idx] = m2.predict_proba(X_tr_dual[va_idx])
    test_d2 += m2.predict_proba(X_te_dual) / skf.n_splits

    m3 = LogisticRegression(C=0.30, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m3.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d3[va_idx] = m3.predict_proba(X_tr_dual[va_idx])
    test_d3 += m3.predict_proba(X_te_dual) / skf.n_splits

    m4 = LogisticRegression(C=0.20, class_weight=tempered_weights, max_iter=1000, random_state=42 + fold)
    m4.fit(X_tr_dense_dual[tr_idx], y_enc[tr_idx])
    oof_d4[va_idx] = m4.predict_proba(X_tr_dense_dual[va_idx])
    test_d4 += m4.predict_proba(X_te_dense_dual) / skf.n_splits

    m5 = CalibratedClassifierCV(RidgeClassifier(alpha=4.0, class_weight=tempered_weights, random_state=42 + fold), cv=3)
    m5.fit(X_tr_dual[tr_idx], y_enc[tr_idx])
    oof_d5[va_idx] = m5.predict_proba(X_tr_dual[va_idx])
    test_d5 += m5.predict_proba(X_te_dual) / skf.n_splits

    m6 = MLPClassifier(hidden_layer_sizes=(160,), alpha=2.0, max_iter=45, early_stopping=True, random_state=42 + fold)
    m6.fit(X_tr_dense_dual[tr_idx], y_enc[tr_idx])
    oof_d6[va_idx] = m6.predict_proba(X_tr_dense_dual[va_idx])
    test_d6 += m6.predict_proba(X_te_dense_dual) / skf.n_splits

print("=" * 74)
print("DUAL-TRANSFORMER (GoEmotions + DistilRoBERTa-Emotion) 5-FOLD CV SUMMARY")
print("=" * 74)
for name, p in [
    ("D1: Calibrated LinearSVC (Balanced, Dual Hybrid)", oof_d1),
    ("D2: Calibrated LinearSVC (Tempered, Dual Hybrid)", oof_d2),
    ("D3: LogisticRegression (Tempered, Dual Hybrid)", oof_d3),
    ("D4: LogisticRegression (Tempered, Dense Dual)", oof_d4),
    ("D5: Calibrated RidgeClassifier (Dual Hybrid)", oof_d5),
    ("D6: Regularized MLP (Dense Dual)", oof_d6),
]:
    print(f"{name:50s} -> Macro-F1: {f1_score(y_enc, p.argmax(axis=1), average='macro'):.4f}")

# Soft-voting ensemble across all 6 diverse models
oof_dual_blend = (
    0.25 * oof_d1
    + 0.25 * oof_d2
    + 0.18 * oof_d3
    + 0.12 * oof_d4
    + 0.12 * oof_d5
    + 0.08 * oof_d6
)
test_dual_blend = (
    0.25 * test_d1
    + 0.25 * test_d2
    + 0.18 * test_d3
    + 0.12 * test_d4
    + 0.12 * test_d5
    + 0.08 * test_d6
)

f1_dual_raw = f1_score(y_enc, oof_dual_blend.argmax(axis=1), average="macro")

# Nested OOF prior-power tuning (evaluated out-of-fold so CV is completely leak-free)
emp_prior = counts / len(y_enc)
oof_nested_preds = np.zeros(len(y_enc), dtype=int)
gammas = []
for tr_idx, va_idx in skf.split(oof_dual_blend, y_enc):
    best_g, best_s = 0.0, -1.0
    for g in np.linspace(-0.30, 0.30, 61):
        s = f1_score(y_enc[tr_idx], (oof_dual_blend[tr_idx] * (emp_prior ** g)).argmax(axis=1), average="macro")
        if s > best_s:
            best_s, best_g = s, g
    gammas.append(best_g)
    oof_nested_preds[va_idx] = (oof_dual_blend[va_idx] * (emp_prior ** best_g)).argmax(axis=1)

f1_dual_nested = f1_score(y_enc, oof_nested_preds, average="macro")
mean_gamma = float(np.mean(gammas))

print("-" * 74)
print(f"6-Model Dual-Transformer Blend (Raw Argmax)      : {f1_dual_raw:.4f} (vs prev 0.6235: {f1_dual_raw - 0.6235:+.4f})")
print(f"6-Model Blend + Nested OOF Prior Scaling (g={mean_gamma:+.3f}): {f1_dual_nested:.4f} (vs prev 0.6235: {f1_dual_nested - 0.6235:+.4f})")
print("-" * 74)

if f1_dual_nested >= f1_dual_raw:
    final_oof = oof_nested_preds
    final_test_preds = le.inverse_transform((test_dual_blend * (emp_prior ** mean_gamma)).argmax(axis=1))
    chosen_desc = f"Dual-Transformer 6-Model Blend + Prior Scaling (CV={f1_dual_nested:.4f})"
else:
    final_oof = oof_dual_blend.argmax(axis=1)
    final_test_preds = le.inverse_transform(test_dual_blend.argmax(axis=1))
    chosen_desc = f"Dual-Transformer 6-Model Blend (CV={f1_dual_raw:.4f})"

print(classification_report(y_true, le.inverse_transform(final_oof), digits=4))

sub_go = pd.DataFrame({
    "id": test_df["post_id"].astype(str),
    "emotion": final_test_preds
})
sub_go.to_csv("submission_goemotions_blend.csv", index=False)
print(f"Saved submission_goemotions_blend.csv ({chosen_desc}), shape={sub_go.shape}")
print("Predicted test distribution:\n", sub_go["emotion"].value_counts())

DUAL-TRANSFORMER (GoEmotions + DistilRoBERTa-Emotion) 5-FOLD CV SUMMARY
D1: Calibrated LinearSVC (Balanced, Dual Hybrid)   -> Macro-F1: 0.6400
D2: Calibrated LinearSVC (Tempered, Dual Hybrid)   -> Macro-F1: 0.6391
D3: LogisticRegression (Tempered, Dual Hybrid)     -> Macro-F1: 0.6351
D4: LogisticRegression (Tempered, Dense Dual)      -> Macro-F1: 0.6313
D5: Calibrated RidgeClassifier (Dual Hybrid)       -> Macro-F1: 0.6414
D6: Regularized MLP (Dense Dual)                   -> Macro-F1: 0.6339
--------------------------------------------------------------------------
6-Model Dual-Transformer Blend (Raw Argmax)      : 0.6415 (vs prev 0.6235: +0.0180)
6-Model Blend + Nested OOF Prior Scaling (g=+0.238): 0.6441 (vs prev 0.6235: +0.0206)
--------------------------------------------------------------------------
              precision    recall  f1-score   support

       anger     0.5775    0.6691    0.6199       674
     disgust     0.5793    0.4185    0.4859       454
        fear     0.

## Final 100% Compliant 9-Family Clean Pipeline